# Optimizacija hiperparametara LightGBM modela

Koristimo RandomizedSearchCV za pronalazenje najboljih parametara za LightGBM model.

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report
import joblib
import warnings
warnings.filterwarnings('ignore')

## 1. Ucitavanje podataka

In [3]:
df = pd.read_csv('preprocessed_data/application_advanced_preprocessed.csv')
print(f"Dimenzije: {df.shape}")

X = df.drop('TARGET', axis=1)
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Dimenzije: (307511, 302)
scale_pos_weight: 11.39


## 2. Bazni model (pre optimizacije)

In [8]:
base_model = lgb.LGBMClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1,
    verbose=-1
)

base_model.fit(X_train_scaled, y_train)
y_proba_base = base_model.predict_proba(X_test_scaled)[:, 1]
base_auc = roc_auc_score(y_test, y_proba_base)
print(f"Bazni ROC-AUC: {base_auc:.6f}")

Bazni ROC-AUC: 0.783701


## 3. RandomizedSearchCV

In [9]:
param_distributions = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [4, 5, 6, 7, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.15, 0.2],
    'num_leaves': [15, 31, 50, 70, 100],
    'min_child_samples': [10, 20, 30, 50, 100],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'reg_alpha': [0, 0.01, 0.1, 1],
    'reg_lambda': [0, 0.01, 0.1, 1],
}

lgb_model = lgb.LGBMClassifier(
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1,
    verbose=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
random_search = RandomizedSearchCV(
    lgb_model,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

print("Pokrecem RandomizedSearchCV (50 iteracija, 5-fold CV)...")
print("Ovo moze potrajati 5-10 minuta...")

random_search.fit(X_train_scaled, y_train)

Pokrecem RandomizedSearchCV (50 iteracija, 5-fold CV)...
Ovo moze potrajati 5-10 minuta...
Fitting 5 folds for each of 50 candidates, totalling 250 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMClassifie...), verbose=-1)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': [0.6, 0.7, ...], 'learning_rate': [0.01, 0.05, ...], 'max_depth': [4, 5, ...], 'min_child_samples': [10, 20, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be

In [11]:
print("Najbolji parametri:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nNajbolji CV ROC-AUC: {random_search.best_score_:.6f}")

best_model_rs = random_search.best_estimator_
y_proba_rs = best_model_rs.predict_proba(X_test_scaled)[:, 1]
rs_auc = roc_auc_score(y_test, y_proba_rs)
print(f"Test ROC-AUC: {rs_auc:.6f}")

improvement_rs = (rs_auc - base_auc) * 100
print(f"Poboljsanje: {improvement_rs:+.4f}%")

Najbolji parametri:
  subsample: 0.9
  reg_lambda: 0
  reg_alpha: 1
  num_leaves: 31
  n_estimators: 500
  min_child_samples: 100
  max_depth: 6
  learning_rate: 0.05
  colsample_bytree: 0.9

Najbolji CV ROC-AUC: 0.783950
Test ROC-AUC: 0.786802
Poboljsanje: +0.3101%


## 4. Fine-tuning oko najboljih parametara

In [12]:
best_params = random_search.best_params_

fine_tune_params = {
    'n_estimators': [best_params['n_estimators'] - 50, best_params['n_estimators'], best_params['n_estimators'] + 50, best_params['n_estimators'] + 100],
    'max_depth': [max(3, best_params['max_depth'] - 1), best_params['max_depth'], best_params['max_depth'] + 1],
    'learning_rate': [best_params['learning_rate'] * 0.5, best_params['learning_rate'], best_params['learning_rate'] * 1.5],
    'num_leaves': [max(10, best_params['num_leaves'] - 10), best_params['num_leaves'], best_params['num_leaves'] + 10],
    'min_child_samples': [max(5, best_params['min_child_samples'] - 10), best_params['min_child_samples'], best_params['min_child_samples'] + 10],
}

fine_model = lgb.LGBMClassifier(
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    reg_alpha=best_params['reg_alpha'],
    reg_lambda=best_params['reg_lambda'],
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=1,
    verbose=-1
)

fine_search = RandomizedSearchCV(
    fine_model,
    param_distributions=fine_tune_params,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

print("Pokrecem fine-tuning (30 iteracija)...")
fine_search.fit(X_train_scaled, y_train)

Pokrecem fine-tuning (30 iteracija)...
Fitting 5 folds for each of 30 candidates, totalling 150 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","LGBMClassifie...9, verbose=-1)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.025, 0.05, ...], 'max_depth': [5, 6, ...], 'min_child_samples': [90, 100, ...], 'n_estimators': [450, 500, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",30
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be us

In [13]:
print("Finalni parametri:")
for param, value in fine_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nNajbolji CV ROC-AUC: {fine_search.best_score_:.6f}")

best_model_final = fine_search.best_estimator_
y_proba_final = best_model_final.predict_proba(X_test_scaled)[:, 1]
final_auc = roc_auc_score(y_test, y_proba_final)
print(f"Test ROC-AUC: {final_auc:.6f}")

improvement_final = (final_auc - base_auc) * 100
print(f"Ukupno poboljsanje: {improvement_final:+.4f}%")

Finalni parametri:
  num_leaves: 31
  n_estimators: 600
  min_child_samples: 90
  max_depth: 7
  learning_rate: 0.025

Najbolji CV ROC-AUC: 0.784521
Test ROC-AUC: 0.787222
Ukupno poboljsanje: +0.3521%


## 5. Rezime i cuvanje modela

In [14]:
results = pd.DataFrame({
    'Model': ['Bazni LightGBM', 'RandomizedSearchCV', 'Fine-tuned'],
    'ROC-AUC': [base_auc, rs_auc, final_auc],
    'Poboljsanje (%)': [0, improvement_rs, improvement_final]
})

print(results.to_string(index=False))

             Model  ROC-AUC  Poboljsanje (%)
    Bazni LightGBM 0.783701         0.000000
RandomizedSearchCV 0.786802         0.310109
        Fine-tuned 0.787222         0.352059


In [15]:
joblib.dump(best_model_final, 'models/lightgbm_optimized.joblib')
joblib.dump(scaler, 'models/scaler_optimized.joblib')

final_params = {**fine_search.best_params_, 
                'subsample': best_params['subsample'],
                'colsample_bytree': best_params['colsample_bytree'],
                'reg_alpha': best_params['reg_alpha'],
                'reg_lambda': best_params['reg_lambda'],
                'scale_pos_weight': scale_pos_weight}

params_df = pd.DataFrame([final_params])
params_df.to_csv('models/best_lightgbm_params.csv', index=False)

print("Sacuvano:")
print("  - models/lightgbm_optimized.joblib")
print("  - models/scaler_optimized.joblib")
print("  - models/best_lightgbm_params.csv")

Sacuvano:
  - models/lightgbm_optimized.joblib
  - models/scaler_optimized.joblib
  - models/best_lightgbm_params.csv


In [16]:
y_pred_final = best_model_final.predict(X_test_scaled)
print("Classification Report (optimizovani model):")
print(classification_report(y_test, y_pred_final, target_names=['Vratio', 'Default']))

Classification Report (optimizovani model):
              precision    recall  f1-score   support

      Vratio       0.96      0.74      0.84     56538
     Default       0.19      0.69      0.30      4965

    accuracy                           0.74     61503
   macro avg       0.58      0.72      0.57     61503
weighted avg       0.90      0.74      0.80     61503

